# Sequential Models: RNN, LSTM & GRU
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Sequential Modeling, Backpropagation Through Time (BPTT), Exploding/Vanishing Gradients, LSTM Gating Mechanics, and GRUs.

---

### 📖 Conceptual Foundations

#### The Sequential Modeling Paradigm

Unlike feedforward networks which assume inputs are independent and identically distributed (i.i.d.), sequential data (text sentences, financial tickers, audio waves, medical ECGs) exhibits strong **temporal dependencies**: the meaning of token $x_t$ depends heavily on the historical context $x_1, \dots, x_{t-1}$. **Recurrent Neural Networks (RNNs)** process sequential inputs step-by-step, maintaining an internal hidden state vector $h_t$ that functions as a running memory.


#### Vanilla RNN Mathematical Formulation

At each time step $t \in \{1, \dots, T\}$:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$\hat{y}_t = \text{softmax}(W_{hy} h_t + b_y)$$

Notice that the weight matrices $W_{hh}, W_{xh}, W_{hy}$ are **shared across all time steps** $t$. Unrolling this operation over $T$ steps produces a computational graph of depth $T$.


#### LSTM Gating Mechanics Breakdown (Hochreiter & Schmidhuber, 1997)

| Gate / State Component | Mathematical Formula | Activation | Functional Role |
| --- | --- | --- | --- |
| Forget Gate ($f_t$) | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | Sigmoid $\in (0, 1)$ | Decides what percentage of old cell memory $C_{t-1}$ to retain ($1$) or wipe clean ($0$). |
| Input Gate ($i_t$) | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | Sigmoid $\in (0, 1)$ | Determines which coordinates of the candidate vector are worth recording into long-term memory. |
| Candidate State ($\tilde{C}_t$) | $\tilde{C}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | Tanh $\in (-1, 1)$ | Creates new candidate information to potentially write into the cell state. |
| Cell State Update ($C_t$) | $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ | Linear Highway | The uninterrupted constant-error carousel carrying long-term memory through time. |
| Output Gate ($o_t$) | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | Sigmoid $\in (0, 1)$ | Controls which portions of the cell state $C_t$ are emitted as the visible hidden state $h_t$. |
| Hidden State ($h_t$) | $h_t = o_t \odot \tanh(C_t)$ | Tanh $\odot$ Sigmoid | The working output passed to the next step and downstream prediction heads. |


#### GRU (Gated Recurrent Unit, Cho et al., 2014)

The **GRU** simplifies LSTM by merging the cell state $C_t$ and hidden state $h_t$ into a single state vector $h_t$, using only **two gates**:
1. **Reset Gate ($r_t$):** Determines how much of past state to ignore.
2. **Update Gate ($z_t$):** Acts simultaneously as forget and input gate ($h_t = (1-z_t) h_{t-1} + z_t \tilde{h}_t$).

**Advantage:** GRUs feature 25% fewer parameters than LSTMs, train faster, and achieve competitive empirical performance on smaller datasets.



In [ ]:
# %matplotlib inline
import numpy as np

def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -15, 15)))

class LSTMCell:
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        # Concatenated weights for [f, i, c, o] gates: (4 * hidden_dim, hidden_dim + input_dim)
        k = hidden_dim + input_dim
        self.W = np.random.randn(4 * hidden_dim, k) * 0.1
        self.b = np.zeros((4 * hidden_dim, 1))
        # Forget gate bias trick: initialize forget bias to 1.0
        self.b[:hidden_dim] = 1.0
        
    def forward(self, x_t, h_prev, c_prev):
        # Concatenate hidden state and input
        concat = np.vstack((h_prev, x_t))
        gates = np.dot(self.W, concat) + self.b
        H = self.hidden_dim
        
        f = sigmoid(gates[0:H])
        i = sigmoid(gates[H:2*H])
        c_cand = np.tanh(gates[2*H:3*H])
        o = sigmoid(gates[3*H:4*H])
        
        c_next = f * c_prev + i * c_cand
        h_next = o * np.tanh(c_next)
        return h_next, c_next

# Run a sequence of 5 tokens
cell = LSTMCell(input_dim=4, hidden_dim=8)
h = np.zeros((8, 1))
c = np.zeros((8, 1))

print('Running 5 time steps of LSTM forward pass:')
for t in range(5):
    x_t = np.random.randn(4, 1)
    h, c = cell.forward(x_t, h, c)
    print(f'Step {t+1}: Hidden state norm = {np.linalg.norm(h):.4f}, Cell state norm = {np.linalg.norm(c):.4f}')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

